# CASE 2 — FEMNIST, natural non-IID, JANUS-FL

Same round protocol as CASE 1, over the writer-disjoint FEMNIST split
(fl/splits.py's writer_disjoint_split), reporting BOTH seen-writer and
unseen-writer accuracy (ROADMAP.md §3.4 -- unseen-writer is the
headline metric). **Run `00_Setup_and_Crypto_Tests.ipynb` and
`CASE1_IID_MNIST.ipynb` first** (CASE1 exercises the full round-loop
wiring end-to-end on the simpler dataset before repeating it here).


In [ ]:
import sys, yaml
sys.path.insert(0, '/content/FL-ZKP')
%cd /content/FL-ZKP

with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)


## Load FEMNIST and compute the writer-disjoint split

In [ ]:
from fl.data_noniid import prepare_case2

femnist_cfg = CFG['datasets']['femnist']
case2_data = prepare_case2(seed=CFG['seed'], cache_dir=femnist_cfg['cache_dir'])
print("clients:", len(case2_data['client_train']))
print("server_root writers:", len(case2_data['split'].server_root_writers))
print("unseen_test writers (headline metric source):", len(case2_data['split'].unseen_test_writers))


## Models: num_classes=62 (digits + upper + lower case)

In [ ]:
from fl.models import build_model, flatten_state_dict

models = {name: build_model(name, num_classes=62, in_channels=1) for name in CFG['mvp_scope']['models']}
flat_weights = {}
for name, m in models.items():
    flat, _ = flatten_state_dict(m)
    flat_weights[name] = flat
    print(name, "param count:", flat.numel())


## Build the ring (one keypair per client) and the trace committee

In [ ]:
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = len(case2_data['client_train'])
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")


## Round loop

Same `fl/round_loop.py::run_training_loop` CASE1 uses (shared, not
duplicated, specifically so the two notebooks can't drift apart --
see that module's docstring and CASE1's "Known gaps" cell for the full
list of what this covers and what's still open, including the sketch
performance wall: **run the smoke test below before any full-scale
sweep**).


In [ ]:
import numpy as np
from fl.round_loop import run_training_loop

SMOKE_ROUNDS = 2
SMOKE_CLIENTS = 8  # must be >= K_MIN (8) in fl/server.py or every round aborts regardless of verification outcome

results = {}
for model_name in CFG['mvp_scope']['models']:
    print(f"\n==== {model_name} (smoke test: {SMOKE_ROUNDS} rounds, {SMOKE_CLIENTS} clients) ====")
    model = models[model_name]
    dim = flat_weights[model_name].numel()

    root_norms = np.random.default_rng(CFG['seed']).normal(5.0, 0.5, size=100)

    final_w_t, history = run_training_loop(
        model=model,
        model_dim=dim,
        client_datasets=case2_data['client_train'][:SMOKE_CLIENTS],
        ring=ring[:SMOKE_CLIENTS],
        client_sks=client_sks[:SMOKE_CLIENTS],
        pairwise_secrets={i: {j: pairwise_secrets[i][j] for j in range(SMOKE_CLIENTS) if j != i} for i in range(SMOKE_CLIENTS)},
        pk_trace=PK_trace,
        server_root_norms=root_norms,
        CFG=CFG,
        rounds=SMOKE_ROUNDS,
    )
    results[model_name] = {'final_w_t': final_w_t, 'history': history}


## Evaluation: seen-writer vs. unseen-writer accuracy

In [ ]:
import torch

def evaluate(model, dataset, model_shapes=None):
    model.eval()
    with torch.no_grad():
        logits = model(dataset.images)
        preds = logits.argmax(dim=1)
        acc = (preds == dataset.labels).float().mean().item()
    return acc

for name, model in models.items():
    seen_accs = [evaluate(model, d) for d in case2_data['client_val']]
    seen_writer_acc = sum(seen_accs) / len(seen_accs) if seen_accs else float('nan')
    unseen_writer_acc = evaluate(model, case2_data['unseen_test'])
    print(f"{name}: seen-writer acc={seen_writer_acc:.3f} (ROADMAP.md target >=0.78), "
          f"unseen-writer acc={unseen_writer_acc:.3f} (target >=0.70, this is the headline number)")
